<a href="https://colab.research.google.com/github/marius-ne/CIE_ProjectB_Group13/blob/junchao/Program_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIE 2025/26 RWTH, PROJECT B, GROUP 13
Junchao Yu, Marius Neuhalfen

ToDo:
- Find defective nodes by comparing perfect structure and imperfect structures for all scenarios
- Group defective nodes into regions (arc sections or track sections)
- Predict whether structure is perfect or imperfect
- Predict where the imperfection lies

There are 25.XXX for deformation and 24.XXX for stress

# Setup

In [ ]:
import os
import sys
import itertools
import functools

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn

from pathlib import Path

In [ ]:
pd.set_option('display.max_columns', 100)

## Data getting (if on Colab)

In [ ]:
import google.colab
google.colab.drive.mount("/content/drive")

Get ancillary data from Github

In [ ]:
!git clone https://github.com/marius-ne/CIE_ProjectB_Group13.git

## Load data

In [ ]:
os.chdir("CIE_ProjectB_Group13")

In [ ]:
os.getcwd()

In [ ]:
!ln -s /content/drive/MyDrive/programB data

In [ ]:
target_folder = "data/Data2"
current_folder = os.getcwd()

if Path(current_folder).name != target_folder:
    os.chdir(Path(current_folder) / Path(target_folder))
print(os.getcwd())


Convert data

In [ ]:
VARIABLES = {
      0: "TotalDeformation.csv",
      1: "DirectionalDeformation_X_axis.csv",
      2: "DirectionalDeformation_Y_axis.csv",
      3: "DirectionalDeformation_Z_axis.csv",
      4: "EquivalentStress.csv",
      5: "ShearStress_XY.csv",
      6: "ShearStress_XZ.csv",
      7: "ShearStress_YZ.csv",
  }
LOADS = {
    0: "Bigger_train",
    1: "Smaller_train",
}
SEASONS = {
    0: "Summer",
    1: "Winter",
}
HEALTHS = {
    0: "Perfect_structure",
    1: "ip_frst_Arc_defect_all_tracks_111",
    2: "ip_1and3track_3_arc_78910",
    3: "ip_first_track_3arc_78910",
    4: "Ip_1track_1_arc_345",
    5: "ip_3track_1_arc_678",
    6: "ip_2_arc_all_tracks_222",
}
TRAIN_CONFIGS = {
    0: "One_train_1st_track",
    1: "One_train_middle_track",
    2: "Two_trains_extreme_track_different_direction",
    3: "Two_trains_extreme_track_same_direction",
}
VARIABLE_NAMES = [var[:-4] for var in VARIABLES.values()]
NODE_NUMBERS = None

def combination_to_string(combination):
    train_config, load, season, health, variable = combination
    return f"{TRAIN_CONFIGS[train_config]}__{LOADS[load]}__{SEASONS[season]}__{HEALTHS[health]}__{VARIABLES[variable][:-4]}"

# Construct list of scenarios (combinations of train configs, loads, seasons, healths, variables)
#   Each scenario is a tuple of (train_config, load, season, health, variable), each encoded
#   as the corresponding key in the dictionaries above
combinations = itertools.product(
        TRAIN_CONFIGS.keys(),
        LOADS.keys(),
        SEASONS.keys(),
        HEALTHS.keys(),
        VARIABLES.keys()
    )
combinations = list(combinations)

# Group by variables, i.e. each group has all variables for one scenario
combinations_grouped_by_variable = [
    combinations[i:i + len(VARIABLES)] for i in range(0, len(combinations), len(VARIABLES))
]
# Group further by healths, i.e. each group has all healths for one scenario (train config, load, season)
combinations_grouped_by_health = [
    combinations_grouped_by_variable[i:i + len(HEALTHS)] for i in range(0, len(combinations_grouped_by_variable), len(HEALTHS))
]



In [ ]:
combinations_grouped_by_health[0]

In [ ]:
def read_data_file(
    train_config: int = 0,
    load: int = 0,
    season: int = 0,
    health: int = 0,
    variable: int = 0,
):
  """Reads data according to format and provides the data-frame as-is, with
  the categorical variables added as columns."""

  # Construct filename from scenario according to the folder structure

  results_paths = ["Results", "Results1"]

  for results_path in results_paths:
      filename = Path()
      filename /= TRAIN_CONFIGS[train_config]
      filename /= LOADS[load]
      filename /= SEASONS[season]
      filename /= HEALTHS[health]
      filename /= results_path
      filename /= VARIABLES[variable]

      if filename.exists():
          break
  else:
      raise FileNotFoundError(f"Data file not found for combination: {combination_to_string((train_config, load, season, health, variable))}")

  # Encode the scenario as categorical columns
  #   -> TODO: Is there a way of encoding that
  #   preserves information? E.g. like encoding the name of a city as its latitude
  df = pd.read_csv(filename)
  num_nodes = len(df)
  df["season"] = season*np.ones(num_nodes,dtype=np.uint8)
  df["health"] = health*np.ones(num_nodes,dtype=np.uint8)
  df["load"] = load*np.ones(num_nodes,dtype=np.uint8)
  df["train_config"] = train_config*np.ones(num_nodes,dtype=np.uint8)

  return df


def get_data():
  """
  Reads all data files from one health group and merges them
  into a single data-frame.
  TODO: Make it read all scenarios, not just one.

  Returns:
      pd.DataFrame: Merged data-frame with all variables as columns.
  """
  global NODE_NUMBERS

  # Select one scenario that has same season, load and trains and goes through
  #   all healths and variables
  scenario = combinations_grouped_by_health[0] # TBD

  # Merge data for all healths in the scenario
  dfs_healths = []
  for same_health_combinations in scenario:
    dfs_variables = []

    #  Merge data for all variables in the current health scenario
    for same_variable_combination in same_health_combinations:
      print("Processing combination:", combination_to_string(same_variable_combination))

      var_name = VARIABLE_NAMES[same_variable_combination[-1]]

      # Get data file for current combination
      df = read_data_file(*same_variable_combination)

      # Turn the variable column into a single one and add a new time column
      df_melted = df.melt(id_vars=["Node Number","season","load","health","train_config"],var_name="variable",value_name=var_name)
      df_melted["time"] = df_melted["variable"].str[-3:].astype(np.float64)
      df_melted.drop(columns=["variable"],inplace=True)

      # Get node numbers and ensure they're consistent
      if NODE_NUMBERS is None:
        NODE_NUMBERS = df_melted["Node Number"].unique()
      else:
        try:
          assert all(NODE_NUMBERS == df_melted["Node Number"].unique())
        except ValueError or AssertionError:
          print("WARNING: Node numbers differ between data files!")
          print("Previous node numbers:", NODE_NUMBERS)
          print("Current node numbers:", df_melted["Node Number"].unique())

      dfs_variables.append(df_melted)

    # Concatenating all variables into a single data frame
    # -> we do an OUTER join, meaning all keys are kept (A U B)
    #   this should be safe, node numbers and the other shared columns are kept
    shared_cols = ["Node Number","season","load","health","train_config","time"]
    df_vars = functools.reduce(lambda left,right: pd.merge(left,right,on=shared_cols,
                                              how='outer'), dfs_variables)
    # Check that data has been preserved
    for df in dfs_variables:
      for var_name in VARIABLE_NAMES:
        if var_name in df.columns:
          merged = pd.merge(df[shared_cols + [var_name]], df_vars[shared_cols + [var_name]],
                            on=shared_cols, how='inner')
          assert len(merged) == len(df)

    dfs_healths.append(df_vars)

  # Check that columns are the same
  assert all(all(df.columns == dfs_healths[0].columns) for df in dfs_healths)

  # Concatenate them together
  df = pd.concat(dfs_healths,ignore_index=True)

  return df

df = get_data()

In [ ]:
df

# Analyze node numbers

Go through all data files and check number of nodes. Create csv file that recaps these.

In [ ]:
nums = []
for ix, combination in enumerate(combinations):
    num_nodes = len(read_data_file(*combination)["Node Number"].unique())
    entry = {}
    entry["train_config"] = combination[0]
    entry["load"] = combination[1]
    entry["season"] = combination[2]
    entry["health"] = combination[3]
    entry["variable"] = combination[4]
    entry["num_nodes"] = num_nodes
    nums.append(entry)
nums

Exemplary plot of node numbers throughout a number of scenarios.

In [ ]:
nn_df = pd.DataFrame(nums)
nn_df.plot(
    x="health", y="num_nodes", kind="bar",
    title=f"Num. nodes per health state"
)

In [ ]:
# Save recap to file
# nn_df.to_csv("node_numbers_Data2_overview.csv", index=False)

# Visualize bridge structure

In [ ]:
# Read tab seperated node export file
node_xyz = pd.read_csv(
    "/content/drive/MyDrive/programB/Data2/nodeExport.txt",
    sep="\t",
    engine="python"
)

In [ ]:
def select_df_subset(combination):
    """Selects a subset of the main data-frame according to the given combination.

    Args:
        combination (tuple): A tuple of (train_config, load, season, health, variable).
    """
    train_config, load, season, health, variable = combination
    var_name = VARIABLE_NAMES[variable]
    df_subset = df[
        (df["train_config"] == train_config) &
        (df["load"] == load) &
        (df["season"] == season) &
        (df["health"] == health)
    ][["Node Number", "time", var_name]]
    return df_subset

In [ ]:
df_subset

In [ ]:
loads

In [ ]:
from ipywidgets import interact, FloatSlider

# Colors for missing nodes
missing_color = 'r'
missing_nn = {}
colors = [
    missing_color if nn in missing_nn.values() else 'b'
    for nn in node_xyz["Node Number"]
]

cmap = plt.get_cmap('viridis')

# Colors for bridge loads
combination = (0, 0, 0, 0, 0)  # Example combination

def plot_bridge_loads_3d_slider(combination):
    """
    Plots bridge loads in 3D with a time slider.
    Args:
        combination (tuple): A tuple of (train_config, load, season, health, variable).
    """
    df_subset = select_df_subset(combination)
    time_points = np.sort(df_subset["time"].unique())
    variable_to_plot = VARIABLE_NAMES[combination[-1]]

    def plot_at_time(time_point_index):
        time_point = time_points[int(time_point_index)]
        timestamp_subset = df_subset[df_subset["time"] == time_point]
        node_loads = [
            timestamp_subset[timestamp_subset["Node Number"] == nn][variable_to_plot].values
            for nn in node_xyz["Node Number"]
        ]
        node_loads_flat = [nl[0] if len(nl) > 0 else np.nan for nl in node_loads]

        fig = plt.figure(figsize=(10,10))
        ax = fig.add_subplot(111, projection='3d')
        scatter = ax.scatter(
            node_xyz["X Location (m)"],
            node_xyz["Y Location (m)"],
            node_xyz["Z Location (m)"],
            c=node_loads_flat, marker='o', s=2,
            cmap="viridis"
        )
        fig.colorbar(scatter, shrink=0.5)
        ax.set_title(f"Bridge Loads at time {time_point}\nFor combination: {combination_to_string(combination)}")
        # fig.savefig(f"../visualization/bridge_loads_3d_{time_point}.png")
        # fig.show()

    interact(
        plot_at_time,
        time_point_index=FloatSlider(
            min=0,
            max=len(time_points)-1,
            step=1,
            value=0,
            description='Time Index'
        )
    )

plot_bridge_loads_3d_slider(combination)

Create GIF from visualizations

In [ ]:
import imageio
import os

# Get all PNG files in the visualization directory, sorted by filename
image_dir = "../visualization/"
image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".png")])


# Read images and create GIF
images = [imageio.imread(os.path.join(image_dir, fname)) for fname in image_files]
gif_path = os.path.join(image_dir, "bridge_loads_animation.gif")
imageio.mimsave(gif_path, images, duration=500)

print(f"GIF saved to {gif_path}")

# Remove NaN

In [ ]:
df.dropna(inplace=True)
df

# Data Inspection

In [ ]:
df.dtypes

In [ ]:
m = 5
df.iloc[:m*10].plot(subplots=True,figsize=(15,15))

In [ ]:
df_sorted = df.sort_values(by=["Node Number","health","time"], ascending=[True, True, True])
# Attention - make sure the indices are reset after sorting
df_sorted.reset_index(drop=True, inplace=True)
df_sorted

In [ ]:
df_sorted[["Node Number","health","time","EquivalentStress"]].iloc[:70].plot(subplots=True,figsize=(10,5))

Compare perfect and imperfect



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.cm as cm
import matplotlib.colors as colors

# ==========================================
# 1. 数据准备 (同前)
# ==========================================

# 确保 df_sorted 存在
if 'df_sorted' not in locals():
    print("正在对数据进行排序...")
    df_sorted = df.sort_values(by=["Node Number","health","time"], ascending=[True, True, True])

# 读取坐标文件
node_path = 'nodeExport.txt'
try:
    node_xyz = pd.read_csv(node_path, sep='\t')
except:
    node_xyz = pd.read_csv(node_path, sep=r'\s+')

# 筛选 Health 0 和 1
df_comp = df_sorted[df_sorted["health"].isin([0, 1])].copy()
target_var = "EquivalentStress"

# ==========================================
# 2. 预计算全局最大值 (用于固定颜色条)
# ==========================================
print("正在预计算全局数据以固定颜色范围...")
global_pivot = df_comp.pivot_table(
    index=["Node Number", "time"],
    columns="health",
    values=target_var
)
global_pivot['diff'] = (global_pivot[1] - global_pivot[0]).abs()
global_max_diff = global_pivot['diff'].max()
print(f"全局最大应力差异: {global_max_diff:.2f} MPa")

# 获取时间步列表
time_steps = np.sort(df_comp['time'].unique())

# ==========================================
# 3. 初始化绘图与静态元素
# ==========================================
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# A. 绘制静态背景 (只画一次，不再更新)
ax.scatter(
    node_xyz['X Location (m)'], node_xyz['Y Location (m)'], node_xyz['Z Location (m)'],
    c='gray', s=1, alpha=0.1, label='Structure'
)

# B. 设置固定的 Colorbar (避免每帧重画)
norm = colors.Normalize(vmin=0, vmax=global_max_diff)
cmap = cm.plasma
mappable = cm.ScalarMappable(norm=norm, cmap=cmap)
mappable.set_array([])
cbar = fig.colorbar(mappable, ax=ax, shrink=0.5, pad=0.05)
cbar.set_label(f'Abs Diff ({target_var})')

# 设置标题、标签、视角
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.view_init(elev=25, azim=-50)

# 用于存储当前帧的高亮散点对象，以便下一帧清除
current_plot = [None]

# ==========================================
# 4. 定义动画更新函数
# ==========================================
def update(frame_idx):
    t = time_steps[frame_idx]

    # 1. 清除上一帧的高亮散点 (如果存在)
    if current_plot[0] is not None:
        current_plot[0].remove()
        current_plot[0] = None

    # 2. 计算当前时刻差异
    # 为了性能，这里只取当前切片
    df_t = df_comp[df_comp['time'] == t]
    pivot_t = df_t.pivot_table(index="Node Number", columns="health", values=target_var)
    diff_series = (pivot_t[1] - pivot_t[0]).abs()

    # 3. 筛选 Top 1%
    if diff_series.empty or diff_series.max() == 0:
        ax.set_title(f"Time: {t:.2f}s (No Data)")
        return

    threshold = diff_series.quantile(0.99)
    significant_nodes = diff_series[diff_series > threshold]

    # 4. 匹配坐标
    highlight_data = node_xyz[node_xyz['Node Number'].isin(significant_nodes.index)].copy()
    highlight_data['diff_val'] = highlight_data['Node Number'].map(significant_nodes)

    # 5. 绘制新的高亮散点
    if not highlight_data.empty:
        p = ax.scatter(
            highlight_data['X Location (m)'],
            highlight_data['Y Location (m)'],
            highlight_data['Z Location (m)'],
            c=highlight_data['diff_val'],
            cmap='plasma',
            s=15,
            alpha=1.0,
            depthshade=False,
            norm=norm # 使用与colorbar相同的归一化
        )
        current_plot[0] = p # 保存对象引用

    ax.set_title(f"Time: {t:.2f}s | Top 1% Diff | Max: {diff_series.max():.2f}")

# ==========================================
# 5. 生成并显示动画
# ==========================================
print(f"正在渲染 {len(time_steps)} 帧动画，请稍候...")

# 创建动画对象
anim = FuncAnimation(fig, update, frames=len(time_steps), interval=200, blit=False)

# 在 Notebook 输出中显示交互式控件
# 这一步会生成 HTML/JS 代码并直接显示
HTML(anim.to_jshtml())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.cm as cm

# ==========================================
# 1. 数据准备
# ==========================================

# 确保 df_sorted 存在
if 'df_sorted' not in locals():
    print("正在对数据进行排序...")
    df_sorted = df.sort_values(by=["Node Number","health","time"], ascending=[True, True, True])

# 读取坐标文件
node_path = 'nodeExport.txt'
try:
    node_xyz = pd.read_csv(node_path, sep='\t')
except:
    node_xyz = pd.read_csv(node_path, sep=r'\s+')

# 筛选 Health 0 和 1
df_comp = df_sorted[df_sorted["health"].isin([0, 1])].copy()
target_var = "EquivalentStress"

# 获取时间步列表
time_steps = np.sort(df_comp['time'].unique())

# ==========================================
# 2. 初始化绘图
# ==========================================
fig = plt.figure(figsize=(10, 7))

# 主图区域 (占据左侧大部分)
ax = fig.add_axes([0.05, 0.05, 0.75, 0.9], projection='3d')

# 颜色条区域 (固定在右侧，避免动画闪烁)
# [left, bottom, width, height]
cax = fig.add_axes([0.85, 0.15, 0.03, 0.7])

# A. 绘制静态背景 (只画一次)
ax.scatter(
    node_xyz['X Location (m)'], node_xyz['Y Location (m)'], node_xyz['Z Location (m)'],
    c='gray', s=1, alpha=0.15, label='Structure'
)

# 设置标题、标签、视角
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.view_init(elev=25, azim=-50)

# 用于存储当前帧的动态对象
current_plot = [None]

# ==========================================
# 3. 定义动画更新函数
# ==========================================
def update(frame_idx):
    t = time_steps[frame_idx]

    # 1. 清除上一帧的高亮散点
    if current_plot[0] is not None:
        current_plot[0].remove()
        current_plot[0] = None

    # 清除上一帧的颜色条内容 (但保留轴的位置)
    cax.clear()

    # 2. 计算当前时刻差异
    df_t = df_comp[df_comp['time'] == t]
    pivot_t = df_t.pivot_table(index="Node Number", columns="health", values=target_var)

    if pivot_t.empty:
        return

    diff_series = (pivot_t[1] - pivot_t[0]).abs()
    current_max = diff_series.max()

    # 3. 筛选 Top 1%
    if diff_series.empty or current_max == 0:
        ax.set_title(f"Time: {t:.2f}s (No Data)")
        return

    threshold = diff_series.quantile(0.99)
    significant_nodes = diff_series[diff_series > threshold]

    # 4. 匹配坐标
    highlight_data = node_xyz[node_xyz['Node Number'].isin(significant_nodes.index)].copy()
    highlight_data['diff_val'] = highlight_data['Node Number'].map(significant_nodes)

    # 5. 绘制新的高亮散点
    if not highlight_data.empty:
        # 注意：这里不设置 vmin/vmax，让它自动适应当前帧的数据范围
        # 这样每一帧最红的点，就代表那一帧的 Max Diff
        p = ax.scatter(
            highlight_data['X Location (m)'],
            highlight_data['Y Location (m)'],
            highlight_data['Z Location (m)'],
            c=highlight_data['diff_val'],
            cmap='plasma',
            s=15,
            alpha=1.0,
            depthshade=False
        )
        current_plot[0] = p # 保存对象引用

        # 6. 重新绘制颜色条 (适应当前范围)
        cbar = fig.colorbar(p, cax=cax)
        cbar.set_label(f'Abs Diff ({target_var})')

    ax.set_title(f"Time: {t:.2f}s | Top 1% | Current Max: {current_max:.2f} MPa")

# ==========================================
# 4. 生成并显示动画
# ==========================================
print(f"正在渲染 {len(time_steps)} 帧自适应颜色动画，请稍候...")

# 创建动画对象
anim = FuncAnimation(fig, update, frames=len(time_steps), interval=200, blit=False)

# 显示动画
HTML(anim.to_jshtml())

In [ ]:
# Filter for health 0 and 1
df_comp = df_sorted[df_sorted["health"].isin([0, 1])]

df_pivot = df_comp.pivot_table(
    index=["Node Number","time","load","train_config","season"],
    columns="health",
    values=VARIABLE_NAMES
)
for variable in VARIABLE_NAMES:
    df_pivot[(variable, 'health_diff')] = df_pivot[(variable, 1)] - df_pivot[(variable, 0)]
df_diff = df_pivot[[ (var, 'health_diff') for var in VARIABLE_NAMES ]]
df_diff.columns = [var for var, _ in df_diff.columns]
df_diff.reset_index(inplace=True)
comp_diff_node_numbers = []
for var in VARIABLE_NAMES:
    for node_number in NODE_NUMBERS:
        if abs(df_diff[var][node_number]) > 0:
            comp_diff_node_numbers.append(node_number)
#df_diff
comp_diff_node_numbers = np.unique(comp_diff_node_numbers)
comp_diff_node_numbers

Isolate nodes with changing variables

In [ ]:
df_time_mean = df_sorted.groupby(by=["Node Number","health"]).mean().reset_index()
df_time_std = df_sorted.groupby(by=["Node Number","health"]).std().reset_index()

In [ ]:

varying_nodes = []
for node_number in NODE_NUMBERS:
    not_same = False
    for var in VARIABLE_NAMES:
        unique_means = df_time_mean[df_time_mean["Node Number"]==node_number][var].unique()
        unique_stds = df_time_std[df_time_std["Node Number"]==node_number][var].unique()
        if len(unique_means) > 1 or len(unique_stds) > 1:
            not_same = True
    if not_same:
        varying_nodes.append(int(node_number))
len(varying_nodes), len(NODE_NUMBERS)

In [ ]:
# Only keep nodes with varying variables
df_varying = df_sorted[df_sorted["Node Number"].isin(varying_nodes)]
df_varying

In [ ]:
df_varying.iloc[:140].plot(subplots=True,figsize=(10,20))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 假设 df_sorted 已经在您的环境中加载 (对应 Notebook Cell 22 之前)
# 如果没有 df_time_mean，我们需要先计算它 (参考 Notebook Cell 24)
# 计算每个节点在不同健康状态下的【时间均值】
df_time_mean = df_sorted.groupby(by=["Node Number", "health"]).mean().reset_index()

# 选择一个敏感变量，例如等效应力
target_var = 'EquivalentStress'

# --- 核心改进步骤 ---

# 1. 提取 Health 0 (健康) 和 Health 1 (缺陷) 的均值数据
# 注意：这里假设您只比较 Health 0 和 1。如果您的缺陷状态是其他数字，请修改 health==1
df_h0 = df_time_mean[df_time_mean['health'] == 0].set_index('Node Number')[target_var]
df_h1 = df_time_mean[df_time_mean['health'] == 1].set_index('Node Number')[target_var]

# 2. 计算【均值的差异绝对值】 (Mean Difference)
# 这比单帧差异更稳定
diff_series = (df_h1 - df_h0).abs()

# 3. 设定阈值：只看差异最大的前 1% (Top 1%)
top_percentage = 0.01
threshold = diff_series.quantile(1 - top_percentage)

# 4. 筛选出显著节点
significant_nodes = diff_series[diff_series > threshold]
print(f"阈值: {threshold:.2f}")
print(f"筛选出 {len(significant_nodes)} 个显著节点 (Top {top_percentage*100}%)")

# 5. 读取坐标 (如果需要)
node_path = 'nodeExport.txt'
try:
    node_xyz = pd.read_csv(node_path, sep='\t')
except:
    node_xyz = pd.read_csv(node_path, sep=r'\s+')

# 6. 匹配坐标并准备绘图
# 仅保留显著节点
plot_data = node_xyz[node_xyz['Node Number'].isin(significant_nodes.index)].copy()
# 将差异值映射给节点，用于上色
plot_data['diff_val'] = plot_data['Node Number'].map(significant_nodes)

# 7. 3D 可视化
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

p = ax.scatter(
    plot_data['X Location (m)'],
    plot_data['Y Location (m)'],
    plot_data['Z Location (m)'],
    c=plot_data['diff_val'],
    cmap='plasma',  # 使用高亮色系
    s=10,
    alpha=1.0
)

fig.colorbar(p, label=f'Mean {target_var} Difference')
ax.set_title(f'Locating Imperfection: Top {top_percentage*100}% Mean Stress Differences')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

plt.show()

#Visualize imperfect nodes of health1

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 1. 读取节点坐标数据
node_path = 'nodeExport.txt'  # 确保路径正确
try:
    node_xyz = pd.read_csv(node_path, sep='\t')
except:
    node_xyz = pd.read_csv(node_path, sep=r'\s+')

# 2. 检查并准备差异节点数据
if 'comp_diff_node_numbers' in locals():
    # 筛选出差异节点
    highlight_nodes = node_xyz[node_xyz['Node Number'].isin(comp_diff_node_numbers)]
    print(f"背景节点数: {len(node_xyz)}")
    print(f"高亮节点数: {len(highlight_nodes)}")
else:
    print("错误: 变量 'comp_diff_node_numbers' 未定义。请先运行计算差异节点的代码。")
    highlight_nodes = pd.DataFrame() # 空数据框以防报错

# 3. 生成叠加 3D 绘图
fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111, projection='3d')

# --- 图层 1: 全桥背景 (Background) ---
# 使用浅灰色、高透明度、小点，勾勒出桥梁整体形状
ax.scatter(
    node_xyz['X Location (m)'],
    node_xyz['Y Location (m)'],
    node_xyz['Z Location (m)'],
    c='gray',  # 浅灰色
    s=1,            # 极小的点
    alpha=0.2,      # 非常透明
    label='Full Structure'
)

# --- 图层 2: 差异节点 (Highlight) ---
# 使用红色、不透明、稍大的点，突出显示
if not highlight_nodes.empty:
    ax.scatter(
        highlight_nodes['X Location (m)'],
        highlight_nodes['Y Location (m)'],
        highlight_nodes['Z Location (m)'],
        c='red',        # 醒目的红色
        s=5,            # 稍大的点
        alpha=1.0,      # 不透明
        depthshade=False, # 关闭深度阴影，保持颜色鲜艳
        label='Difference Nodes (comp_diff_node_numbers)'
    )

# 设置标签和视角
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Overlay: Imperfection Nodes on Bridge Structure')
ax.legend()

# 调整视角以获得更好的观察效果 (可选)
# ax.view_init(elev=20, azim=-45)

plt.show()

# Figure out bridge structure

NOTE: This is now obsolete because we have the node locations.

In [ ]:
len(NODE_NUMBERS)

In [ ]:
from ipywidgets import interact, IntSlider

def integer_factors(n):
    """Returns the list of integer factors of n."""
    factors = []
    for i in range(1, n + 1):
        if n % i == 0:
            factors.append(i)
    return factors
def plot_integer_widths(node_numbers_of_interest: list[int]):
    """Plots the bridge structure as images for all possible widths."""
    img_widths = integer_factors(len(NODE_NUMBERS))
    for w in img_widths:
        img = np.zeros((w,len(NODE_NUMBERS)//w))
        flat_img = img.flatten()
        for i,node_number in enumerate(NODE_NUMBERS):
            if node_number in node_numbers_of_interest:
                flat_img[i] = 1
        img = flat_img.reshape(img.shape)
        plt.figure(figsize=(5,5))
        plt.imshow(img, cmap='gray', interpolation='nearest')
        plt.title(f'Node Variation Map (Width: {w})')
        plt.xlabel('Node Index')
        plt.ylabel('Node Index')
        plt.show()
def plot_load_by_widths_interactive(df, timestep):
    img_widths = integer_factors(len(NODE_NUMBERS))
    times = df["time"].unique()
    if timestep not in times:
        raise ValueError(f"Timestep {timestep} not found in data. Available times: {times}")
    df_time = df[df["time"] == timestep]
    def plot_at_width(width_idx):
        w = img_widths[width_idx]
        img = np.zeros((w, len(NODE_NUMBERS)//w))
        flat_img = img.flatten()
        for i, node_number in enumerate(NODE_NUMBERS):
            load_value = df_time[df_time["Node Number"] == node_number]["TotalDeformation"].values
            assert len(load_value) <= 1, f"Multiple load values found for node {node_number} at time {timestep}"
            if len(load_value) == 1:
                flat_img[i] = load_value[0]
        img = flat_img.reshape(img.shape)
        plt.figure(figsize=(5,5))
        plt.imshow(img, cmap='viridis', interpolation='nearest')
        plt.title(f'Bridge Load Map at time {timestep} (Width: {w}), (Height: {len(NODE_NUMBERS)//w})')
        plt.xlabel('Node Index')
        plt.ylabel('Node Index')
        plt.show()
    interact(plot_at_width, width_idx=IntSlider(min=0, max=len(img_widths)-1, step=1, value=0, description='Width Index'))

def plot_load_by_time_interactive(df, width):
    times = np.sort(df["time"].unique())
    def plot_at_time(time_idx):
        time = times[time_idx]
        df_time = df[df["time"] == time]
        img = np.zeros((width, len(NODE_NUMBERS)//width))
        flat_img = img.flatten()
        for i, node_number in enumerate(NODE_NUMBERS):
            load_value = df_time[df_time["Node Number"] == node_number]["TotalDeformation"].values
            assert len(load_value) <= 1, f"Multiple load values found for node {node_number} at time {time}"
            if len(load_value) == 1:
                flat_img[i] = load_value[0]
        img = flat_img.reshape(img.shape)
        plt.figure(figsize=(5,5))
        plt.imshow(img, cmap='viridis', interpolation='nearest')
        plt.title(f'Bridge Load Map at time {time} (Width: {width})')
        plt.xlabel('Node Index')
        plt.ylabel('Node Index')
        plt.show()
    interact(plot_at_time, time_idx=IntSlider(min=0, max=len(times)-1, step=1, value=0, description='Time Index'))


plot_load_by_widths_interactive(df_sorted[df_sorted["health"]==0], timestep=0.1)
plot_load_by_time_interactive(df_sorted[df_sorted["health"]==0], width=34)

In [ ]:
2210/(42*3)

In [ ]:
plot_integer_widths(comp_diff_node_numbers)

# Training

In [ ]:
df_train = df_varying.copy()

In [ ]:
X_raw, y_raw = df_train.drop(columns=["health"]), df_train["health"]
X_train_raw, X_test_raw, y_train, y_test = sklearn.model_selection.train_test_split(
    X_raw, y_raw, test_size = 0.1, random_state = 0, shuffle=True,
)

In [ ]:
def standardize(X_train_raw, X_test_raw):
    scaler = sklearn.preprocessing.StandardScaler()

    scaler.fit(X_train_raw)
    X_train = scaler.transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    return X_train, X_test

X_train, X_test = standardize(X_train_raw, X_test_raw)

Decision Tree

In [ ]:
# model = sklearn.tree.DecisionTreeClassifier()
# model.fit(X_train, y_train)

In [ ]:
# model.score(X_test, y_test), model.score(X_train, y_train)

Neural Network

In [ ]:
# model = sklearn.neural_network.MLPClassifier(hidden_layer_sizes=(1000,1000,1000),verbose=1)
# model.fit(X_train, y_train)

In [ ]:
# model.score(X_test, y_test), model.score(X_train, y_train)